## Import libraries

In [12]:
import llm_funcs.funcs as llm_helpers

from transformers import DPRContextEncoder, DPRContextEncoderTokenizer, DPRQuestionEncoder, DPRQuestionEncoderTokenizer

import random
import faiss

## Read text file used as context

In [2]:
paragraphs = llm_helpers.read_and_split_text('source_files/companyPolicies.txt')

random.seed(42)
random.shuffle(paragraphs)

## Process context

LLMs work by tokenizing and embedding the given text into vector representations that can later be searched for relevant queried information. In this project, I'm using a Dense Passage Retriever (DPR) model to encode passages.

In [3]:
# Define context parameters
MAX_LEN = 256 # Max numnber of tokens

In [ ]:
# Context tokenization
ctxt_tokenizer = DPRContextEncoderTokenizer.from_pretrained('facebook/dpr-ctx_encoder-single-nq-base')

# Context embedding
ctxt_encoder = DPRContextEncoder.from_pretrained('facebook/dpr-ctx_encoder-single-nq-base')

ctxt_embeddings = llm_helpers.encode_contexts(paragraphs, ctxt_tokenizer, ctxt_encoder, MAX_LEN)

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 18860.49it/s]
[transformers] DPRContextEncoder LOAD REPORT from: facebook/dpr-ctx_encoder-single-nq-base
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
ctx_encoder.bert_model.pooler.dense.bias   | UNEXPECTED |  | 
ctx_encoder.bert_model.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


I'm using the Facebook AI Similarity Search (FAISS) to identify index embedded contexts and efficiently search them for similarities to an user's query.

In [8]:
# Define FAISS parameters
EMBEDDING_DIMS = 768 # Matches dimenions of embeddings defined by DPR model

In [11]:
# Create FAISS index for the embeddings - consider converting embeddings to float32
index = faiss.IndexFlatL2(EMBEDDING_DIMS)
index.add(ctxt_embeddings)

## Process queries

In [ ]:
# Question tokenization
qstn_tokenizer = DPRQuestionEncoderTokenizer.from_pretrained('facebook/dpr-question_encoder-single-nq-base')

# Question embedding
qstn_encoder = DPRQuestionEncoder.from_pretrained('facebook/dpr-question_encoder-single-nq-base')
